# ESI Triage Inference, Parsing, Extraction

We run 150 vignettes through four different prompting conditions (zero-shot, system-safety, few-shot, and chain-of-thought) through the Llama 3 8B model for inference in this notebook.  

Two outputs are saved in particular:
- `results/responses.jsonl`: the generated text along with the parsed ESI level and the outcome for each row (correct, undertriage, overtriage).
- `results/hidden_states.npz`: Captures the last 4 layer activations at last token position (snapshot of what model "knows" about patient); hidden-state extraction.

## Setup Drive, Libraries, HF

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers accelerate

In [3]:
from huggingface_hub import login
login()  # interactive prompt — paste token when asked

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Paths, Configs, Import

In [4]:
# set paths, model, and configurations for loop
import sys
from pathlib import Path

drive_path = Path("/content/drive/My Drive/triage")
results_dir = drive_path / "results"
data_file = drive_path / "data/esi_vignettes.json"

sys.path.insert(0, str(drive_path))
results_dir.mkdir(parents=True, exist_ok=True)

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
conditions = ["zero_shot", "system_safety", "few_shot", "chain_of_thought"]
n_hidden_layers = 4
max_new_tokens  = 512
BATCH_SIZE = 4
RESUME = False

In [5]:
# import and GPU check

import json
import re
import time

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from prompt_templates import build_prompt

print(f"torch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
  print(f"GPU: {torch.cuda.get_device_name(0)}")

torch 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Helper functions

In [6]:
# load vignettes
def load_vignettes(path):
  with open(path) as f:
    return json.load(f)

# parse ESI level
def parse_esi_level(text):
  """Extract ESI level integer from 'ESI_LEVEL: X' in response."""
  match = re.search(r"ESI_LEVEL:\s*([1-5])", text)
  return int(match.group(1)) if match else None

# classifying triage level
def classify_triage(predicted, gold):
  """Return 'correct', 'undertriage', 'overtriage', or 'unparsed'."""
  if predicted is None:
    return "unparsed"
  if predicted == gold:
    return "correct"
  return "undertriage" if predicted > gold else "overtriage"

# chat template
def build_chat_input(tokenizer, prompt):
  messages = [
    {"role": "system", "content": prompt["system"]},
    {"role": "user",   "content": prompt["user"]},
  ]
  text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
  )
  return tokenizer(text, return_tensors="pt")

# hidden state extraction function
def extract_hidden_states(model, input_ids, attention_mask, n_layers):
  """Forward pass only that returns (n_layers, hidden_dim) at last token."""
  with torch.no_grad():
    outputs = model(
      input_ids=input_ids,
      attention_mask=attention_mask,
      output_hidden_states=True,
    )
  layer_vecs = [
    outputs.hidden_states[-(i + 1)][0, -1, :].cpu().float().numpy()
    for i in range(n_layers - 1, -1, -1)
  ]
  return np.stack(layer_vecs)

# resume logic (if run disconnects; don't need to train from scratch)
def load_completed_keys(responses_file):
  if not responses_file.exists():
    return set()
  keys = set()
  with open(responses_file) as f:
    for line in f:
      try:
        row = json.loads(line)
        keys.add(f"{row['vignette_id']}|{row['condition']}")
      except (json.JSONDecodeError, KeyError):
        pass
  return keys

print("helpers ready")

helpers ready


## Inference Loop

In [9]:
# add padding and load model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
  model_name,
  torch_dtype=torch.float16,
  device_map="auto",
)
model.eval()

print(f"hidden_dim: {model.config.hidden_size}")
print(f"n_layers: {model.config.num_hidden_layers}")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

device: cuda


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

hidden_dim: 4096
n_layers: 32
VRAM used: 16.1 GB


In [10]:
# load vignettes, define result paths
vignettes = load_vignettes(data_file)
responses_path = results_dir / "responses.jsonl"
hs_path  = results_dir / "hidden_states.npz"

# (resume logic)
completed = load_completed_keys(responses_path) if RESUME else set()
hs_store  = {}
if RESUME and hs_path.exists():
  existing = np.load(hs_path, allow_pickle=False)
  hs_store = {k: existing[k] for k in existing.files}

total = len(vignettes) * len(conditions)
print(f"{total} runs total, {len(completed)} already done")

# open jsonl file for writing
responses_fh = open(responses_path, "a" if RESUME else "w")
t0   = time.time()
done = 0

try:
    # outer loop over conditions for batches to share same system prompt structure
  for condition in conditions:
    pending = [v for v in vignettes
                if f"{v['vignette_id']}|{condition}" not in completed]

    for i in range(0, len(pending), BATCH_SIZE):
      batch = pending[i : i + BATCH_SIZE]

      # tokenize as a padded batch
      texts = [
        tokenizer.apply_chat_template(
          [{"role": "system", "content": p["system"]},
            {"role": "user",   "content": p["user"]}],
          tokenize=False, add_generation_prompt=True
        )
        for p in [build_prompt(condition, v["vignette_text"]) for v in batch]
      ]
      inputs = tokenizer(texts, return_tensors="pt", padding=True).to(device)
      input_ids = inputs["input_ids"]
      attention_mask = inputs["attention_mask"]
      prompt_len = input_ids.shape[1]

      # generate output with set configurations
      with torch.no_grad():
        output_ids = model.generate(
          input_ids=input_ids,
          attention_mask=attention_mask,
          max_new_tokens=max_new_tokens,
          do_sample=False,
          pad_token_id=tokenizer.eos_token_id,
        )

      # hidden states based on one forward pass for the whole batch
      # left-padding means pos[-1] is last real token for each sequence
      with torch.no_grad():
        hs_out = model(
          input_ids=input_ids,
          attention_mask=attention_mask,
          output_hidden_states=True,
        )
      batch_hidden = np.stack([
        np.stack([
          hs_out.hidden_states[-(j + 1)][b, -1, :].cpu().float().numpy()
          for j in range(n_hidden_layers - 1, -1, -1)
        ])
        for b in range(len(batch))
      ])

      # decode and record each item in the batch
      for b_idx, vignette in enumerate(batch):
        vid  = vignette["vignette_id"]
        gold = vignette["esi_gold_label"]

        response_text = tokenizer.decode(
          output_ids[b_idx, prompt_len:], skip_special_tokens=True
        )

        # debugging statement (was used fixing some batching issues)
        if done < 4:
          print("DEBUG:", repr(response_text[:300]))

        hs_store[f"{vid}__{condition}"] = batch_hidden[b_idx]

        predicted = parse_esi_level(response_text)
        outcome   = classify_triage(predicted, gold)

        # record
        responses_fh.write(json.dumps({
          "vignette_id":    vid,
          "condition":      condition,
          "esi_gold_label": gold,
          "predicted_esi":  predicted,
          "outcome":        outcome,
          "response_text":  response_text,
        }) + "\n")
        responses_fh.flush()

        done += 1
        elapsed = time.time() - t0
        rate = done / elapsed
        remaining = (total - len(completed) - done) / rate if rate > 0 else 0
        print(
          f"[{done:>3}/{total - len(completed)}] "
          f"{vid} / {condition:<22} "
          f"gold={gold} pred={predicted} {outcome:<12} "
          f"ETA {remaining/60:.1f}m"
        )

finally:
  responses_fh.close()
  if hs_store:
    np.savez(hs_path, **hs_store)
    print(f"\nsaved hidden_states.npz  ({len(hs_store)} entries)")
  print(f"saved responses.jsonl    ({done} new rows)")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


600 runs total, 0 already done
DEBUG: "Based on the patient's presentation, I would assign an ESI level as follows:\n\nThe patient is experiencing eye pain and discomfort, which is concerning, but it is not a life-threatening emergency. The patient is able to communicate effectively and is not experiencing any other concerning symptoms suc"
[  1/600] CH9_001 / zero_shot              gold=5 pred=3 overtriage   ETA 69.0m
DEBUG: "Based on the patient's presentation, I would assign the following reasoning:\n\nThe patient has a history of depression and has attempted self-harm by cutting her wrists, which is a concerning sign of potential suicidal ideation. Her respiratory rate is slow, which may indicate respiratory depression,"
[  2/600] CH9_002 / zero_shot              gold=1 pred=1 correct      ETA 34.4m
DEBUG: "Based on the patient's presentation, I would assign an ESI level as follows:\n\nThe patient is unresponsive, which indicates a severe neurological impairment. The Glasgow Coma S

## Per-condition summary

In [11]:
summary = {c: {"correct": 0, "undertriage": 0, "overtriage": 0, "unparsed": 0}
           for c in conditions}
counts = {c: 0 for c in conditions}

with open(responses_path) as f:
  for line in f:
    row = json.loads(line)
    c = row["condition"]
    summary[c][row["outcome"]] += 1
    counts[c] += 1

print(f"{'condition':<22} {'n':>4}  {'correct':>8}  {'undertriage':>12}  {'overtriage':>11}  {'unparsed':>9}")
print("-" * 75)
for c in conditions:
  n = counts[c]
  if n == 0:
    continue
  s = summary[c]
  print(
    f"{c:<22} {n:>4}  "
    f"{s['correct']:>7} ({100*s['correct']/n:4.1f}%)  "
    f"{s['undertriage']:>10} ({100*s['undertriage']/n:4.1f}%)  "
    f"{s['overtriage']:>9} ({100*s['overtriage']/n:4.1f}%)  "
    f"{s['unparsed']:>7} ({100*s['unparsed']/n:4.1f}%)"
  )

print(f"\nOutputs saved to: {results_dir}")

condition                 n   correct   undertriage   overtriage   unparsed
---------------------------------------------------------------------------
zero_shot               150       66 (44.0%)          36 (24.0%)         48 (32.0%)        0 ( 0.0%)
system_safety           150       55 (36.7%)          24 (16.0%)         71 (47.3%)        0 ( 0.0%)
few_shot                150       85 (56.7%)          19 (12.7%)         46 (30.7%)        0 ( 0.0%)
chain_of_thought        150       41 (27.3%)          78 (52.0%)         24 (16.0%)        7 ( 4.7%)

Outputs saved to: /content/drive/My Drive/triage/results
